# PREDICTION: Churn

In [0]:
%pip install feature-engine
%pip install lightgbm

In [0]:
import mlflow
import os
notebook_path = os.getcwd()

In [0]:
# Champion model
model = mlflow.sklearn.load_model("models:/workspace.default.churn_lightgbm_balanced_tuned@champion")

model

In [0]:
# importing new data
df_inferece = spark.sql(""" 
SELECT
    *
FROM workspace.default.feature_store_inference
""").toPandas()

df_inferece.head(5)

In [0]:
# Checking nulls
df_inferece.isnull().sum().sort_values(ascending=False)


In [0]:
# Features
X_inference = df_inferece[model.feature_names_in_]

# Confere que a classe 0 (churn) é mesmo a primeira coluna do predict_proba
assert list(model.classes_) == [0, 1], model.classes_

predict = model.predict_proba(X_inference)[:, 1]

df_inferece['ProbaChurn'] = predict



In [0]:
df_inferece.head()

In [0]:

spark.sql("DROP TABLE IF EXISTS workspace.default.output_churn")

spark_df = spark.createDataFrame(df_inferece[['customerID', 'ProbaChurn']])

# Salve como uma tabela Delta permanente
spark_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.output_churn")


In [0]:
# df_to_save = df.sort_values(by='ProbaChurn', ascending=False).head(50)

# df_to_save.to_csv(os.path.join(notebook_path, '../predictions/predictions.csv'), index=False)

import datetime

# Use this code to save the prediction to a csv file for submission:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
df_inferece.to_csv(os.path.join(notebook_path, f'../predictions/predictionPCT_Group3_{timestamp}.csv'), index=False)

In [0]:
# Defining cut off

# Feature Importance Analysis (LightGBM Champion)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import metrics

tb_abt2 = 'raw_abt2_Churn'

df = spark.table(f'workspace.default.{tb_abt2}').toPandas()

features = [
    # --- 1. Customer Demographics (Numeric Binary) ---
    'isMale',
    'SeniorCitizen',
    'isPartner',
    'HasDependents',
    'MonthlyIncome',

    # --- 2. Account & Tenure ---
    'nbrTenure',                  # Continuous tenure
    'isNewCustomer',              # High-risk early churn flag (tenure <= 6)
    'isPaperlessBilling',
    'isContractOneYear',          # Baseline = Month-to-month
    'isContractTwoYear',

    # --- 3. Payment Methods (One flag dropped as reference category) ---
    'isPaymentMethod_ElectronicCheck',  # Known highest-risk payment method
    'isPaymentMethod_MailedCheck',
    'isPaymentMethod_BankTransfer',
    'isPaymentMethod_CreditCard',

    # --- 4. Core Telecom Services ---
    'hasPhoneService',
    'hasMultipleLines',
    'hasInternetService',
    'isInternetService_DSL',
    'isInternetService_Fiber',          # Known high churn driver in EDA

    # --- 5. Add-on Internet Services ---
    'hasOnlineSecurity',
    'hasOnlineBackup',
    'hasDeviceProtection',
    'hasTechSupport',
    'hasStreamingTV',
    'hasStreamingMovies',
    'nbrInternetServicesAddOnQty',      # Total count of add-ons
    'hasSecurityBundle',                # High-loyalty flag

    # --- 6. High-Stakes Business Interactions (The Revenue Levers) ---
    'isM2M_dual',                       # The #1 revenue hemorrhage segment (47.3% churn)
    'isM2M_internetOnly',
    'isOneYear_dual',
    'isTwoYear_dual',

    # --- 7. Billing & Financial Ratios ---
    'MonthlyCharges',
    'TotalCharges',                     # Imputed by your champion Ridge model
    'TotalSpend',                       # Interaction (tenure * MonthlyCharges)
    'chargeRatio',                      # MonthlyCharges / MonthlyIncome (affordability stress)
    'numServices',                      # Service breadth
    'chargesPerService'                 # Value perception
]

from sklearn import model_selection

target = 'TargetChurn'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y,
                                                                    random_state=42,
                                                                    stratify=y,
                                                                    test_size=0.2)


print(f'Size Train dataset: {y_train.shape[0]} ({y_train.shape[0]/y.shape[0]*100:.2f}%). Target ratio: ({y_train.mean()*100:.2f}%)')

print(f'Size Test dataset: {y_test.shape[0]} ({y_test.shape[0]/y.shape[0]*100:.2f}%). Target ratio: ({y_test.mean()*100:.2f}%)')


# If using the pipeline returned by train_and_log_churn:
champion_pipeline = model

# Threshold Tuning & Trade-Off Analysis

# 1. Predict raw churn probabilities on Test Set
y_test_probs = champion_pipeline.predict_proba(X_test[features])[:, 1]
base_rate = float(y_train.mean())  # ~0.2669

# 2. Evaluate metrics across thresholds from 0.05 to 0.90
thresholds = np.linspace(0.05, 0.90, 86)
records = []

for t in thresholds:
    y_pred_t = (y_test_probs >= t).astype(int)
    
    rec = metrics.recall_score(y_test, y_pred_t, zero_division=0)
    prec = metrics.precision_score(y_test, y_pred_t, zero_division=0)
    f1 = metrics.f1_score(y_test, y_pred_t, zero_division=0)
    acc = metrics.accuracy_score(y_test, y_pred_t)
    
    # % of total customers flagged for retention intervention
    pct_flagged = y_pred_t.mean() * 100
    
    records.append({
        'Threshold': round(t, 3),
        'Recall': round(rec, 4),
        'Precision': round(prec, 4),
        'F1_Score': round(f1, 4),
        'Accuracy': round(acc, 4),
        'Customers_Flagged_Pct': round(pct_flagged, 2)
    })

df_thresholds = pd.DataFrame(records)

# 3. Find key benchmark points
best_f1_row = df_thresholds.loc[df_thresholds['F1_Score'].idxmax()]
target_recall_row = df_thresholds[df_thresholds['Recall'] >= 0.75].sort_values(by='F1_Score', ascending=False).iloc[0]
base_rate_row = df_thresholds.iloc[(df_thresholds['Threshold'] - base_rate).abs().argsort()[:1]].iloc[0]

print("=== KEY THRESHOLD BENCHMARKS ===")
print(f"1. Base Rate Cutoff ({base_rate_row['Threshold']}): Recall = {base_rate_row['Recall']*100:.1f}%, Precision = {base_rate_row['Precision']*100:.1f}%, F1 = {base_rate_row['F1_Score']:.4f}")
print(f"2. Best F1 Cutoff ({best_f1_row['Threshold']}): Recall = {best_f1_row['Recall']*100:.1f}%, Precision = {best_f1_row['Precision']*100:.1f}%, F1 = {best_f1_row['F1_Score']:.4f}")
print(f"3. Max Precision for Recall >= 75% ({target_recall_row['Threshold']}): Recall = {target_recall_row['Recall']*100:.1f}%, Precision = {target_recall_row['Precision']*100:.1f}%, Flagged = {target_recall_row['Customers_Flagged_Pct']}%\n")

# 4. Plot Precision, Recall, and F1 vs. Threshold
plt.figure(figsize=(10, 6))
plt.plot(df_thresholds['Threshold'], df_thresholds['Recall'], label='Recall (Churn Detected)', color='navy', lw=2)
plt.plot(df_thresholds['Threshold'], df_thresholds['Precision'], label='Precision (True Churners in Outreach)', color='darkorange', lw=2)
plt.plot(df_thresholds['Threshold'], df_thresholds['F1_Score'], label='F1-Score (Harmonic Mean)', color='green', lw=2)

# Benchmark lines
plt.axvline(0.50, color='gray', linestyle=':', label='Default Cutoff (0.50)')
plt.axvline(base_rate, color='red', linestyle='--', label=f'Base Rate Cutoff ({base_rate:.2f})')
plt.axhline(0.75, color='black', linestyle='-.', alpha=0.6, label='PRD Target Recall (>= 0.75)')

plt.title("Light GBM (balanced): Threshold Tuning: Precision vs. Recall vs. F1", fontsize=14, fontweight='bold')
plt.xlabel("Probability Decision Threshold", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.xlim(0.05, 0.90)
plt.ylim(0, 1.02)
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.25), ncol=3)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()





In [0]:
print(f'Best cut-off: {best_f1_row["Threshold"]}')

import datetime
import numpy as np
import pandas as pd

# 1. Threshold → Yes/No
DEPLOY_THRESHOLD = 0.29
df_inferece['prediction'] = (df_inferece['ProbaChurn'] >= DEPLOY_THRESHOLD).map({True: 'Yes', False: 'No'})

# 2. Keep ONLY customerID + prediction
submission = df_inferece[['customerID', 'prediction']].copy()

# 3. Sanity check (~37.6% Yes expected)
print(submission['prediction'].value_counts(normalize=True))
print(submission.head())

# Use this code to save the prediction to a csv file for submission:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission.to_csv(os.path.join(notebook_path, f'../predictions/predictionPCT_Group3_{timestamp}.csv'), index=False)



In [0]:
import datetime

def prediction_function(input_df):
    '''
    An example model function, that just predicts randomly whether a customer will churn.
    TODO: Make a better model.
    '''
    X = input_df[['customerID']].copy()
    X['prediction'] = np.random.uniform(size=len(X)) >= 0.5
    X['prediction'] = X['prediction'].map({True: 'Yes', False: 'No'})
    return X

import os
candidates = ['../../06_data/inference.csv', '06_data/inference.csv', 'inference.csv', '../../../inference.csv']
inference_path = next((p for p in candidates if os.path.exists(p)), '06_data/inference.csv')
test_df = pd.read_csv(inference_path)
prediction = prediction_function(test_df)
print(prediction.head().transpose())
# Use this code to save the prediction to a csv file for submission:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
prediction.to_csv(os.path.join(notebook_path, f'../predictions/prediction_Gt_{timestamp}.csv'))

In [0]:
df.head()

## Feature Importance

In [0]:
import pandas as pd

estimator = model[-1]                      # remove last step of the pipeline
names = model.feature_names_in_            # features that go into the model

assert len(names) == len(estimator.feature_importances_), "Features names and importance with different sizes"

df_importance = (
    pd.DataFrame({"Feature": names, "Importance": estimator.feature_importances_})
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(df_importance)


In [0]:

# Feature Importance Cumulativa
df_importance["Acumulado"] = df_importance["Importance"].cumsum()

display(df_importance)

In [0]:
# Saving Feature Importance

df_importance.to_csv(os.path.join(notebook_path, "../../05_outputs/df_importances.csv"), index=False)

